# MoE Load Balancing Loss — 面试版

**难度：** Medium

**面试要点：** 不用 torch.softmax，手写 softmax + 无 for 循环计算 f

In [ ]:
import torch

In [ ]:
# ✅ INTERVIEW

def moe_load_balance_loss(router_logits, num_experts):
    N = router_logits.shape[0]

    # ---- 手写 softmax ----
    max_val = router_logits.max(dim=-1, keepdim=True).values
    shifted = router_logits - max_val
    exp_vals = torch.exp(shifted)
    probs = exp_vals / exp_vals.sum(dim=-1, keepdim=True)

    # ---- 无 for 循环计算 f ----
    # 用 one-hot 编码 + mean 代替 for 循环
    assignments = router_logits.argmax(dim=-1)  # (N,)
    # one_hot: (N, num_experts)，每行只有一个 1
    one_hot = torch.zeros(N, num_experts, device=router_logits.device)
    one_hot.scatter_(1, assignments.unsqueeze(1), 1.0)
    # f = one_hot.mean(dim=0) → (num_experts,)
    f = one_hot.mean(dim=0)

    # P = probs.mean(dim=0)
    P = probs.mean(dim=0)

    return num_experts * (f * P).sum()

In [ ]:
num_experts = 4
N_tokens = 100
logits_uniform = torch.zeros(N_tokens, num_experts)
loss = moe_load_balance_loss(logits_uniform, num_experts)
print(f"Uniform routing loss: {loss.item():.4f}  (expected 1.0)")

In [ ]:
from torch_judge import check
check("moe_load_balance")

## 📝 核心思路总结

1. **scatter_ 实现 one-hot**：`one_hot.scatter_(1, idx.unsqueeze(1), 1.0)`
2. **无循环计数**：one_hot.mean(dim=0) 等价于逐个比较
3. **手写 softmax**：减最大值 → exp → 归一化